# Hybrid Machine Learning & Optimization: Garment Employee Productivity
This notebook demonstrates the integration of Machine Learning (Random Forest) and Mixed Integer Programming (PuLP) for supply chain optimization. 

**Objectives:**
1. Perform Exploratory Data Analysis (EDA) on the Garment Employee Productivity dataset (UCI ID: 597).
2. Train a Random Forest Regressor to predict worker productivity.
3. Use SHAP to interpret the trained model.
4. Formulate and solve a Mixed Integer Programming (MIP) model using PuLP to optimally allocate workers based on predicted productivities.

## 1. Environment Setup
Install all required libraries to ensure the notebook runs end-to-end.

In [ ]:
# Install the required libraries. 
# Depending on the environment, you might need to run this cell first.
!pip install -q ucimlrepo pandas numpy matplotlib seaborn scikit-learn shap pulp

## 2. Importing Libraries and Loading Data
We fetch the Garment Employee Productivity dataset directly from the UCI Machine Learning Repository.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import shap
import pulp

# Fetch dataset
productivity_prediction_of_garment_employees = fetch_ucirepo(id=597)

X = productivity_prediction_of_garment_employees.data.features
y = productivity_prediction_of_garment_employees.data.targets

# Combine into a single DataFrame for easier EDA
df = pd.concat([X, y], axis=1)

# Display the first few rows
df.head()

## 3. Exploratory Data Analysis (EDA)
In this section, we will inspect the data structure, check for missing values and duplicates, and summarize the statistics.

### 3.1. Dataset Information
Display the general structure, column data types, and non-null counts.

In [ ]:
# Display DataFrame information
df.info()

### 3.2. Missing Values Analysis
Calculate the count and percentage of missing values per column.

In [ ]:
# Check for missing values
missing_counts = df.isna().sum()
missing_percent = (missing_counts / len(df)) * 100

missing_summary = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage (%)': missing_percent
})

# Display columns with missing values
missing_summary[missing_summary['Missing Count'] > 0]

### 3.3. Unique Values per Column
Check the number of unique values in each column to understand cardinality.

In [ ]:
# Count unique values for each column
df.nunique()

### 3.4. Duplicate Rows Check
Check for any duplicate rows in the dataset.

In [ ]:
# Check for duplicate rows
duplicates_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates_count}")

### 3.5. Descriptive Statistics
Generate a statistical summary for all columns (both numeric and categorical).

In [ ]:
# Descriptive statistics
df.describe(include='all')

## 4. Data Visualizations
We will create business-relevant plots to better understand the variables.

### 4.1. Distribution of Target Variable (`actual_productivity`)
Understanding the spread and central tendency of actual productivity.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['actual_productivity'], kde=True, bins=30, color='blue')
plt.title('Distribution of Actual Productivity')
plt.xlabel('Actual Productivity')
plt.ylabel('Frequency')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

### 4.2. Productivity by Department
Comparing productivity levels across the different departments. Note: 'sweing' is a typo in the original dataset for 'sewing', and sometimes contains trailing spaces. We will fix spaces in preprocessing but visualize raw data first.

In [ ]:
plt.figure(figsize=(8, 5))
# Using strip to group correctly if there are whitespace variations
sns.boxplot(x=df['department'].str.strip(), y=df['actual_productivity'], palette='Set2')
plt.title('Productivity Comparison Across Departments')
plt.xlabel('Department')
plt.ylabel('Actual Productivity')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

### 4.3. Targeted vs. Actual Productivity
Scatter plot to see how often actual productivity meets or exceeds the targeted productivity. The diagonal red line represents `y = x`.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x='targeted_productivity', y='actual_productivity', data=df, alpha=0.6)

# Diagonal line
max_val = max(df['targeted_productivity'].max(), df['actual_productivity'].max())
plt.plot([0, max_val], [0, max_val], color='red', linestyle='--', label='Target Met (y=x)')

plt.title('Targeted vs Actual Productivity')
plt.xlabel('Targeted Productivity')
plt.ylabel('Actual Productivity')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

### 4.4. Effect of Incentive on Productivity
Exploring if higher incentives generally lead to higher productivity.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x='incentive', y='actual_productivity', hue=df['department'].str.strip(), data=df, alpha=0.7)
plt.title('Effect of Incentive on Actual Productivity')
plt.xlabel('Incentive')
plt.ylabel('Actual Productivity')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

## 5. Data Preprocessing
Based on the EDA, we will clean and format the data before modeling.

### 5.1. Handle Missing Values
The `wip` (Work in Progress) column has over 40% missing values. As requested, we will impute these with the median value. If there were other missing values under 5%, we would drop them, but only `wip` is missing here.

In [ ]:
# Impute missing values in 'wip' with the median
wip_median = df['wip'].median()
df['wip'] = df['wip'].fillna(wip_median)

# Verify no more missing values
print("Remaining missing values:", df.isna().sum().sum())

### 5.2. Clean Categorical Columns
1. We strip whitespaces from the `department` column.
2. We remove rows with 'Quarter5' in the `quarter` column as it might represent an anomaly (a year typically has 4 quarters, and Quarter5 contains very few rows).

In [ ]:
# Strip whitespaces from department names
df['department'] = df['department'].str.strip()

# Remove 'Quarter5'
df = df[df['quarter'] != 'Quarter5']

print("Unique quarters after cleaning:", df['quarter'].unique())
print("Unique departments after cleaning:", df['department'].unique())

### 5.3. Drop Irrelevant or High-Cardinality Columns
The `date` column is complex to handle for a simple model. We will drop it. We also ensure no leakage variables are created for training.

In [ ]:
# Drop 'date' column
df = df.drop(columns=['date'])

### 5.4. One-Hot Encoding
Convert categorical features (`quarter`, `department`, `day`) into numerical formats using one-hot encoding.

In [ ]:
# Identify categorical columns
cat_cols = ['quarter', 'department', 'day']

# Apply One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Ensure all boolean columns are converted to int (optional, but good for some models)
df_encoded = df_encoded.astype({col: 'int' for col in df_encoded.select_dtypes(include='bool').columns})

df_encoded.head()

### 5.5. Train-Test Split
Split the dataset into 80% training and 20% testing data.

In [ ]:
# Separate features (X) and target (y)
X_model = df_encoded.drop(columns=['actual_productivity'])
y_model = df_encoded['actual_productivity']

# 80/20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_model, y_model, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

## 6. Machine Learning Modeling & Evaluation
Train a Random Forest Regressor and optimize it using GridSearchCV.

### 6.1. Hyperparameter Tuning using GridSearchCV
We define a parameter grid for the Random Forest and find the best combination.

In [ ]:
# Define the model
rf = RandomForestRegressor(random_state=42)

# Define parameter grid for tuning
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

# Setup GridSearchCV
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, 
                           cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

# Fit the model
grid_search.fit(X_train, y_train)

# Get the best model
best_rf = grid_search.best_estimator_

print(f"Best Parameters: {grid_search.best_params_}")

### 6.2. Model Evaluation
Evaluate the best model on the unseen test set using MAE, MSE, and R2 Score.

In [ ]:
# Predict on test set
y_pred = best_rf.predict(X_test)

# Calculate metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"R-squared (R2 Score): {r2:.4f}")

### 6.3. Actual vs Predicted Plot
Visualize how well the model predictions align with the actual test values.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.7)

# Perfect prediction diagonal line
max_val = max(y_test.max(), y_pred.max())
plt.plot([0, max_val], [0, max_val], color='red', linestyle='--')

plt.title('Random Forest: Actual vs Predicted Productivity')
plt.xlabel('Actual Productivity')
plt.ylabel('Predicted Productivity')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

## 7. Model Interpretation with SHAP
Explain the model's predictions and feature importance using SHAP (SHapley Additive exPlanations).

### 7.1. Compute SHAP Values
Using `shap.TreeExplainer` on the best Random Forest model.

In [ ]:
# Initialize JS visualization for SHAP (if running in a notebook interface)
shap.initjs()

# Create explainer
explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_test)

### 7.2. SHAP Summary Bar Plot
Global feature importance: showing which features have the highest average impact on predictions.

In [ ]:
# Bar plot for global feature importance
shap.summary_plot(shap_values, X_test, plot_type="bar")

### 7.3. SHAP Detailed Summary Plot
Shows direction of impact: e.g., do higher values of a feature increase or decrease productivity?

In [ ]:
# Detailed summary plot
shap.summary_plot(shap_values, X_test)

### 7.4. Business Interpretation
* **Incentive / Targeted Productivity**: If these are at the top, it means setting goals and rewarding employees heavily influences final productivity.
* **SMV (Standard Minute Value)**: Represents the time allocated for a task. Higher SMV might be correlated with lower productivity or vice versa.
* **WIP (Work in Progress)**: Shows how backlog affects output.

## 8. Optimization Model (MIP) with PuLP
Now, we use the predicted productivity to optimally allocate workers across departments.

**Problem Statement**:
Maximize total productivity by allocating workers to the 'sweing' and 'finishing' departments.

**Coefficients**: Average *predicted* productivity for each department (from our Machine Learning model evaluated on the entire dataset).

**Constraints**:
1. Total workers available ≤ 100
2. Maximum workers per department = 60
3. Minimum workers per department = 20

### 8.1. Calculate Objective Coefficients
We predict the productivity for all rows, append it to the dataset, and calculate the mean predicted productivity per department.

In [ ]:
# Predict productivity for the entire dataset using the trained model
df_encoded['predicted_productivity'] = best_rf.predict(X_model)

# The original 'department' column was one-hot encoded as 'department_sweing'.
# If 'department_sweing' == 1, it's 'sweing', else it's 'finishing'.
sweing_avg_pred = df_encoded[df_encoded['department_sweing'] == 1]['predicted_productivity'].mean()
finishing_avg_pred = df_encoded[df_encoded['department_sweing'] == 0]['predicted_productivity'].mean()

print(f"Average Predicted Productivity for Sweing: {sweing_avg_pred:.4f}")
print(f"Average Predicted Productivity for Finishing: {finishing_avg_pred:.4f}")

### 8.2. Formulate and Solve the MIP Model
Set up decision variables, objective function, and constraints in PuLP.

In [ ]:
# Initialize the maximization problem
prob = pulp.LpProblem("Worker_Allocation_Optimization", pulp.LpMaximize)

# Decision Variables: Integer number of workers in each department
workers_sweing = pulp.LpVariable("Workers_Sweing", lowBound=0, cat='Integer')
workers_finishing = pulp.LpVariable("Workers_Finishing", lowBound=0, cat='Integer')

# Objective Function: Maximize Total Expected Productivity
prob += (sweing_avg_pred * workers_sweing) + (finishing_avg_pred * workers_finishing), "Total_Productivity"

# Constraints
prob += workers_sweing + workers_finishing <= 100, "Total_Workers_Constraint"
prob += workers_sweing <= 60, "Max_Capacity_Sweing"
prob += workers_finishing <= 60, "Max_Capacity_Finishing"
prob += workers_sweing >= 20, "Min_Workers_Sweing"
prob += workers_finishing >= 20, "Min_Workers_Finishing"

# Solve the problem
solver_status = prob.solve()

### 8.3. Display and Evaluate MIP Results
Analyze the output of the solver.

In [ ]:
# Display Status
print(f"Solver Status: {pulp.LpStatus[solver_status]}")

# Display Objective Value
total_max_productivity = pulp.value(prob.objective)
print(f"Maximum Total Productivity Value: {total_max_productivity:.4f}")

# Display Optimal Allocation
optimal_sweing = workers_sweing.varValue
optimal_finishing = workers_finishing.varValue

print("\n--- Optimal Worker Allocation ---")
print(f"Sweing Department: {optimal_sweing} workers")
print(f"Finishing Department: {optimal_finishing} workers")

### 8.4. Business Interpretation of Optimization
**Explanation of Solver Status**:
The solver status indicates whether an 'Optimal' solution was found, or if the constraints made it 'Infeasible'. 

**Business Recommendation**:
Based on the Random Forest predictions, one department yields a higher average productivity per worker. The optimization model logically attempts to max out the capacity of the highest-performing department (up to the limit of 60) and allocates the remaining workers (up to the total limit of 100) to the other department, whilst respecting the minimum 20 workers per department rule. 

For example, if "Finishing" has higher average predicted productivity, the business should allocate exactly the maximum allowed (60 workers) to Finishing, and the remaining 40 workers to Sweing. This guarantees maximum possible output while maintaining operational minimums.